In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Test Iceberg Local Notebook")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://warehouse/iceberg")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
print(spark.version)

3.5.0


In [2]:
#Tạo database bhxh
spark.sql("""
    CREATE DATABASE IF NOT EXISTS local.bhxh
""")

DataFrame[]

In [3]:
#Tạo bảng masters
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.bhxh.masters (
        SO_SO_BHXH STRING,
        HO_TEN STRING,
        GIOI_TINH STRING,
        MA_TINH STRING,
        TUOI INT
    )
    USING iceberg
""")

DataFrame[]

In [4]:
#Hiển thị các bảng có trong database bhxh
spark.sql("""
    SHOW TABLES IN local.bhxh
""").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     bhxh|  masters|      false|
+---------+---------+-----------+



In [5]:
spark.sql("""
    INSERT INTO local.bhxh.masters VALUES
    ('BH000001', 'Nguyen Van An', 'M', '01', 25),
    ('BH000002', 'Tran Thi Binh', 'F', '02', 31),
    ('BH000003', 'Le Van Cuong', 'M', '01', 42),
    ('BH000004', 'Pham Thi Dung', 'F', '03', 28),
    ('BH000005', 'Hoang Van Em', 'M', '04', 35)
""")


DataFrame[]

In [6]:
spark.sql("""
    SELECT * FROM local.bhxh.masters
""").show()

+----------+-------------+---------+-------+----+
|SO_SO_BHXH|       HO_TEN|GIOI_TINH|MA_TINH|TUOI|
+----------+-------------+---------+-------+----+
|  BH000001|Nguyen Van An|        M|     01|  25|
|  BH000002|Tran Thi Binh|        F|     02|  31|
|  BH000003| Le Van Cuong|        M|     01|  42|
|  BH000004|Pham Thi Dung|        F|     03|  28|
|  BH000005| Hoang Van Em|        M|     04|  35|
+----------+-------------+---------+-------+----+



In [7]:
#Xem các snapshot trong table masters
spark.sql("""
    SELECT *
    FROM local.bhxh.masters.snapshots
""").show()

+--------------------+------------------+---------+---------+--------------------+--------------------+
|        committed_at|       snapshot_id|parent_id|operation|       manifest_list|             summary|
+--------------------+------------------+---------+---------+--------------------+--------------------+
|2026-08-14 10:21:...|434264843984627010|     NULL|   append|s3a://warehouse/i...|{spark.app.id -> ...|
+--------------------+------------------+---------+---------+--------------------+--------------------+



In [8]:
#Thêm dữ liệu mới
spark.sql("""
     INSERT INTO local.bhxh.masters VALUES
    ('BH0000010', 'Nguyen Van Duc', 'M', '01', 23),
    ('BH0000012', 'Nguyen Thi Anh', 'F', '09', 22)
""")

DataFrame[]

In [9]:
#Xem snapshot mới, sau khi thay đổi (thêm mới)
spark.sql("""
    SELECT *
    FROM local.bhxh.masters.snapshots
""").show()

+--------------------+-------------------+------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|         parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+------------------+---------+--------------------+--------------------+
|2026-08-14 10:21:...| 434264843984627010|              NULL|   append|s3a://warehouse/i...|{spark.app.id -> ...|
|2026-08-14 10:21:...|8063328108136724679|434264843984627010|   append|s3a://warehouse/i...|{spark.app.id -> ...|
+--------------------+-------------------+------------------+---------+--------------------+--------------------+



In [10]:
#Hiển thị snapshot
spark.sql("""
    SELECT
        snapshot_id,
        committed_at
    FROM local.bhxh.masters.snapshots;
""").show()

+-------------------+--------------------+
|        snapshot_id|        committed_at|
+-------------------+--------------------+
| 434264843984627010|2026-08-14 10:21:...|
|8063328108136724679|2026-08-14 10:21:...|
+-------------------+--------------------+



In [13]:
#Time travel: hiển thị bảng masters với snapshot ID (snapshot đầu tiên) => thông tin ban đầu insert
spark.sql("""
    SELECT *
    FROM local.bhxh.masters
    VERSION AS OF 434264843984627010;
""").show()

+----------+-------------+---------+-------+----+
|SO_SO_BHXH|       HO_TEN|GIOI_TINH|MA_TINH|TUOI|
+----------+-------------+---------+-------+----+
|  BH000001|Nguyen Van An|        M|     01|  25|
|  BH000002|Tran Thi Binh|        F|     02|  31|
|  BH000003| Le Van Cuong|        M|     01|  42|
|  BH000004|Pham Thi Dung|        F|     03|  28|
|  BH000005| Hoang Van Em|        M|     04|  35|
+----------+-------------+---------+-------+----+



In [14]:
#Time travel: hiển thị bảng masters với snapshot ID (snapshot sau khi thêm) => data ban đầu + data mới
spark.sql("""
    SELECT *
    FROM local.bhxh.masters
    VERSION AS OF 8063328108136724679;
""").show()

+----------+--------------+---------+-------+----+
|SO_SO_BHXH|        HO_TEN|GIOI_TINH|MA_TINH|TUOI|
+----------+--------------+---------+-------+----+
| BH0000010|Nguyen Van Duc|        M|     01|  23|
| BH0000012|Nguyen Thi Anh|        F|     09|  22|
|  BH000001| Nguyen Van An|        M|     01|  25|
|  BH000002| Tran Thi Binh|        F|     02|  31|
|  BH000003|  Le Van Cuong|        M|     01|  42|
|  BH000004| Pham Thi Dung|        F|     03|  28|
|  BH000005|  Hoang Van Em|        M|     04|  35|
+----------+--------------+---------+-------+----+



In [15]:
spark.sql("""
    SELECT *
    FROM local.bhxh.masters
    ORDER BY HO_TEN
""").show()

+----------+--------------+---------+-------+----+
|SO_SO_BHXH|        HO_TEN|GIOI_TINH|MA_TINH|TUOI|
+----------+--------------+---------+-------+----+
|  BH000005|  Hoang Van Em|        M|     04|  35|
|  BH000003|  Le Van Cuong|        M|     01|  42|
| BH0000012|Nguyen Thi Anh|        F|     09|  22|
|  BH000001| Nguyen Van An|        M|     01|  25|
| BH0000010|Nguyen Van Duc|        M|     01|  23|
|  BH000004| Pham Thi Dung|        F|     03|  28|
|  BH000002| Tran Thi Binh|        F|     02|  31|
+----------+--------------+---------+-------+----+



In [16]:
#Xóa master có so_so_bhxh = 'BH000001'
spark.sql("""
    DELETE FROM local.bhxh.masters
    WHERE SO_SO_BHXH = 'BH000001';
""")

DataFrame[]

In [17]:
#Hiển thị kết quả xem đã xóa chưa
spark.sql("""
    select * from local.bhxh.masters
""").show()

+----------+--------------+---------+-------+----+
|SO_SO_BHXH|        HO_TEN|GIOI_TINH|MA_TINH|TUOI|
+----------+--------------+---------+-------+----+
| BH0000010|Nguyen Van Duc|        M|     01|  23|
|  BH000002| Tran Thi Binh|        F|     02|  31|
| BH0000012|Nguyen Thi Anh|        F|     09|  22|
|  BH000003|  Le Van Cuong|        M|     01|  42|
|  BH000004| Pham Thi Dung|        F|     03|  28|
|  BH000005|  Hoang Van Em|        M|     04|  35|
+----------+--------------+---------+-------+----+



In [18]:
#Hiển thị các snapshot để thấy có snapshot mới (delete)
spark.sql("""
    select * from local.bhxh.masters.snapshots
""").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-08-14 10:21:...| 434264843984627010|               NULL|   append|s3a://warehouse/i...|{spark.app.id -> ...|
|2026-08-14 10:21:...|8063328108136724679| 434264843984627010|   append|s3a://warehouse/i...|{spark.app.id -> ...|
|2026-08-14 10:22:...|9095376715964719478|8063328108136724679|   delete|s3a://warehouse/i...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [19]:
#Update tên của master BH000002
spark.sql("""
    update local.bhxh.masters
    set HO_TEN = "Tran Thanh Binh"
    where SO_SO_BHXH = "BH000002"
""")

DataFrame[]

In [20]:
#Hiển thị để check xem master BH000002 đã thay đổi tên chưa
spark.sql("""
    select * from local.bhxh.masters
""").show()

+----------+---------------+---------+-------+----+
|SO_SO_BHXH|         HO_TEN|GIOI_TINH|MA_TINH|TUOI|
+----------+---------------+---------+-------+----+
|  BH000002|Tran Thanh Binh|        F|     02|  31|
| BH0000010| Nguyen Van Duc|        M|     01|  23|
|  BH000003|   Le Van Cuong|        M|     01|  42|
| BH0000012| Nguyen Thi Anh|        F|     09|  22|
|  BH000004|  Pham Thi Dung|        F|     03|  28|
|  BH000005|   Hoang Van Em|        M|     04|  35|
+----------+---------------+---------+-------+----+



In [21]:
#Hiển thị các snapshots để kierm tra snapshot mới vừa upadete master BH000002 (overwrite)
spark.sql("""
    select * from local.bhxh.masters.snapshots
""").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-08-14 10:21:...| 434264843984627010|               NULL|   append|s3a://warehouse/i...|{spark.app.id -> ...|
|2026-08-14 10:21:...|8063328108136724679| 434264843984627010|   append|s3a://warehouse/i...|{spark.app.id -> ...|
|2026-08-14 10:22:...|9095376715964719478|8063328108136724679|   delete|s3a://warehouse/i...|{spark.app.id -> ...|
|2026-08-14 10:22:...|3568269335664262754|9095376715964719478|overwrite|s3a://warehouse/i...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [22]:
#Schema Evolution: thêm cột mới
spark.sql("""
    alter table local.bhxh.masters
    add column NGHE_NGHIEP string;
""")

DataFrame[]

In [23]:
#Hiển thị dữ liệu khi thêm cột mới
spark.sql("""
    select * from local.bhxh.masters
""").show()

+----------+---------------+---------+-------+----+-----------+
|SO_SO_BHXH|         HO_TEN|GIOI_TINH|MA_TINH|TUOI|NGHE_NGHIEP|
+----------+---------------+---------+-------+----+-----------+
|  BH000002|Tran Thanh Binh|        F|     02|  31|       NULL|
| BH0000010| Nguyen Van Duc|        M|     01|  23|       NULL|
|  BH000003|   Le Van Cuong|        M|     01|  42|       NULL|
| BH0000012| Nguyen Thi Anh|        F|     09|  22|       NULL|
|  BH000004|  Pham Thi Dung|        F|     03|  28|       NULL|
|  BH000005|   Hoang Van Em|        M|     04|  35|       NULL|
+----------+---------------+---------+-------+----+-----------+



In [24]:
#Partition: partitoned theo mã tỉnh để khi lọc theo mã tỉnh sẽ nhanh hơn
spark.sql("""
    create table local.bhxh.masters_partitioned(
        SO_SO_BHXH string,
        HO_TEN string,
        GIOI_TINH string,
        MA_TINH long,
        TUOI int
    )using iceberg
    partitioned by (MA_TINH)
""")

DataFrame[]

In [25]:
#Thêm thông tin để test partitioned
spark.sql("""
    INSERT INTO local.bhxh.masters_partitioned VALUES
    ('BH000001', 'Nguyen Van An', 'M', 01, 25),
    ('BH000002', 'Tran Thi Binh', 'F', 02, 31),
    ('BH000003', 'Le Van Cuong', 'M', 01, 42),
    ('BH000004', 'Pham Thi Dung', 'F', 03, 28),
    ('BH000005', 'Hoang Van Em', 'M', 04, 35),
    ('BH000006', 'Pham Thi Dung 2', 'F', 03, 28),
    ('BH000007', 'Hoang Van Em 2', 'M', 04, 35)
""")

DataFrame[]

In [26]:
#hiển thị các bảng có trong database bhxh
spark.sql("""
    SHOW TABLES IN local.bhxh
""").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|     bhxh|            masters|      false|
|     bhxh|masters_partitioned|      false|
+---------+-------------------+-----------+

